# Network Planning Tower — Data Story
## RTBB1015 · Beefsteak 15lb 39ct No1 · MPL1 Shortage Crisis

This notebook tells a complete supply chain story in five acts:

| Act | Chart | Business Question |
|-----|-------|-------------------|
| The Macro View | Chart 1 | Which commodities are in shortage, and when? |
| The Fire Alarm | Chart 2 | Why is RTBB1015 at MPL1 critically short? (root cause) |
| The Overnight Detective | Chart 2.5 | Which POs failed to arrive overnight — and how many cases are missing? |
| The Allocation Solution | Chart 3 | How should we split the 2,000-case emergency resupply across customers? |
| The Financial Impact | Chart 4 | What is the dollar value of tier-based vs flat-cut allocation? |

**Key facts:**
- SKU: `RTBB1015` — Beefsteak 15lb 39ct Beef No1
- Location: `MPL1` — peak shortage date 2026-03-24 (−1,710 cases)
- Root cause: 715 inbound cases never received at MPL2_INV + MPL6-B (overall fill rate 53.1%)
- Emergency resupply modelled: 2,000 cases vs 5,071 unfulfilled backorders
- Overnight exceptions (2026-03-17): 61 POs short, 59,573 cases missing

**Run order:** Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5 → Cell 5b → Cell 5c → Cell 6 → Cell 7

In [ ]:
# ── Cell 1 · Imports ──────────────────────────────────────────────────────────
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# ── Cell 2 · Load curated data, merge, calculate Net_Position ─────────────────
#
# Reads the curated CSVs from data/curated/, joins them into a single
# analysis-ready DataFrame, computes Net_Position, and saves the result as
# data/curated/Intermediate_Master_View.csv.

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
CURATED = os.path.normpath(os.path.join(NOTEBOOK_DIR, "..", "..", "data", "curated"))

def load(name):
    df = pd.read_csv(os.path.join(CURATED, f"{name}.csv"))
    print(f"  {name}: {len(df):,} rows, {len(df.columns)} cols")
    return df

print("Loading curated CSVs ...")
fact_inv  = load("Fact_Inventory")
fact_ib   = load("Fact_Inbound")
fact_dem  = load("Fact_Demand")
dim_prod  = load("Dim_Product")
dim_loc   = load("Dim_Location")

# Normalise date key column names so all facts share "DateKey"
fact_ib  = fact_ib.rename(columns={"DeliveryDateKey": "DateKey"})
fact_dem = fact_dem.rename(columns={"ShipDateKey": "DateKey"})

# Aggregate Fact_Inbound to DateKey x SKUKey x LocationKey grain
ib_agg = (
    fact_ib
    .groupby(["DateKey", "SKUKey", "LocationKey"], as_index=False)
    .agg(
        Inbound_ExpectedQty = ("ExpectedQty", "sum"),
        Inbound_ReceivedQty = ("ReceivedQty", "sum"),
    )
)

# Aggregate Fact_Demand to DateKey x SKUKey x LocationKey grain
dem_agg = (
    fact_dem
    .groupby(["DateKey", "SKUKey", "LocationKey"], as_index=False)
    .agg(
        Demand_OriginalQty    = ("OriginalQty",    "sum"),
        Demand_OutstandingQty = ("OutstandingQty", "sum"),
    )
)

# Outer-join all facts on shared grain (Fact_Inventory is the spine)
join_keys = ["DateKey", "SKUKey", "LocationKey"]
merged = (
    fact_inv
    .merge(ib_agg,  on=join_keys, how="outer")
    .merge(dem_agg, on=join_keys, how="outer")
)

# Attach dimension descriptors
merged = merged.merge(
    dim_prod[["SKUKey", "SKUCode", "ProductDescription", "Commodity", "ProductGroup"]],
    on="SKUKey", how="left"
)
merged = merged.merge(
    dim_loc[["LocationKey", "LocationCode", "LocationType"]],
    on="LocationKey", how="left"
)

# Fill missing numerics with 0
num_cols = merged.select_dtypes(include="number").columns
merged[num_cols] = merged[num_cols].fillna(0)

# Net_Position = (InventoryOnHand + Inbound_ReceivedQty + TransfersInReceived)
#              - (GoodSalesShipping + TransferShippingOut + Donations)
merged["Net_Position"] = (
    (merged["InventoryOnHand"] + merged["Inbound_ReceivedQty"] + merged["TransfersInReceived"])
    - (merged["GoodSalesShipping"] + merged["TransferShippingOut"] + merged["Donations"])
)
merged["Shortage_Flag"] = (merged["Net_Position"] < 0).astype(int)
merged["Date"] = pd.to_datetime(merged["DateKey"].astype(int).astype(str), format="%Y%m%d")

display_cols = [
    "Date", "SKUCode", "ProductDescription", "Commodity", "ProductGroup",
    "LocationCode", "LocationType",
    "InventoryOnHand", "Inbound_ExpectedQty", "Inbound_ReceivedQty",
    "TransfersInExpected", "TransfersInReceived", "TotalSupply",
    "GoodSalesShipping", "Demand_OriginalQty", "Demand_OutstandingQty",
    "Donations", "TransferShippingOut",
    "Net_Position", "Shortage_Flag", "NetAvailable",
]
df_viz = merged[[c for c in display_cols if c in merged.columns]].copy()

out_path = os.path.join(CURATED, "Intermediate_Master_View.csv")
df_viz.to_csv(out_path, index=False)

print(f"\nMaster View: {len(df_viz):,} rows x {len(df_viz.columns)} cols")
print(f"Shortage rows: {df_viz['Shortage_Flag'].sum():,} of {len(df_viz):,}")
print(f"Saved -> {out_path}")
df_viz.head()

In [ ]:
# ── Cell 3 · Build Allocation Scenario DataFrame ─────────────────────────────
#
# Scenario: Emergency resupply of 2,000 cases must be allocated across
# 5,071 unfulfilled backorders for RTBB1015 — the highest-shortage SKU.
#
# Context: RTBB1015 at MPL1 accumulated -5,071 cases of shortage during
# 2026-03-22 to 2026-03-27 because expected inbound at MPL2_INV (524 cases)
# and MPL6-B (191 cases) was never received (0% fill rate at both locations).
# An emergency spot purchase of 2,000 cases has been secured.
#
# Tier rules:
#   Tier 1 - High Margin  : 0% cut   (contractual / highest-margin accounts)
#   Tier 2 - Standard     : 0% cut   (standard retail with SLA commitments)
#   Tier 3 - Flexible/Spot: 100% cut (non-binding spot and promo buyers)
#
# Math: Tier 1 (1,200) + Tier 2 (800) = 2,000 (exactly matches supply)
#       Tier 3 (3,071) = deferred to next resupply = 3,071 cases cut
#       Total cuts = 3,071 = shortage  (validates correctly)

SKU_CONTEXT   = "RTBB1015 - Beefsteak 15lb 39ct Beef No1"
TOTAL_DEMAND  = 5_071   # cumulative unfulfilled backorder cases
TOTAL_SUPPLY  = 2_000   # emergency spot-purchase allocation
SHORTAGE      = TOTAL_DEMAND - TOTAL_SUPPLY   # 3,071 cases
FLAT_CUT_RATE = SHORTAGE / TOTAL_DEMAND       # 60.6%

CUT_RATE = {
    "Tier 1 - High Margin":    0.00,
    "Tier 2 - Standard":       0.00,
    "Tier 3 - Flexible/Spot":  1.00,
}

UNIT_PRICE = {
    "Tier 1 - High Margin":    24.00,
    "Tier 2 - Standard":       20.00,
    "Tier 3 - Flexible/Spot":  17.00,
}

raw = [
    # Tier 1 - contractual / highest-margin (total: 1,200 cases)
    ("Metro Ontario Inc",          "Tier 1 - High Margin",    450),
    ("Kroger National",            "Tier 1 - High Margin",    500),
    ("Walmart Premium Account",    "Tier 1 - High Margin",    250),
    # Tier 2 - standard retail with SLA (total: 800 cases)
    ("Meijer Inc",                 "Tier 2 - Standard",       300),
    ("Sobeys Inc",                 "Tier 2 - Standard",       250),
    ("Loblaws Regional",           "Tier 2 - Standard",       250),
    # Tier 3 - flexible / spot (total: 3,071 cases — fully deferred)
    ("Discount Foods East",        "Tier 3 - Flexible/Spot",  600),
    ("Food Service Distributor A", "Tier 3 - Flexible/Spot",  700),
    ("Promo / Secondary Buyer",    "Tier 3 - Flexible/Spot",  500),
    ("Spot Market Buyer",          "Tier 3 - Flexible/Spot",  571),
    ("Clearance Account",          "Tier 3 - Flexible/Spot",  700),
]

df_alloc = pd.DataFrame(raw, columns=["Customer_Name", "Priority_Tier", "Original_Order_Qty"])

df_alloc["Suggested_Cut_Qty"]   = (df_alloc["Original_Order_Qty"] * df_alloc["Priority_Tier"].map(CUT_RATE)).round().astype(int)
df_alloc["Final_Allocated_Qty"] = df_alloc["Original_Order_Qty"] - df_alloc["Suggested_Cut_Qty"]
df_alloc["Unit_Price"]          = df_alloc["Priority_Tier"].map(UNIT_PRICE)
df_alloc["Flat_Allocated_Qty"]  = (df_alloc["Original_Order_Qty"] * (1 - FLAT_CUT_RATE)).round().astype(int)
df_alloc["Revenue_Saved"]       = ((df_alloc["Final_Allocated_Qty"] - df_alloc["Flat_Allocated_Qty"]) * df_alloc["Unit_Price"]).round(2)
df_alloc["Fill_Rate_Pct"]       = (df_alloc["Final_Allocated_Qty"] / df_alloc["Original_Order_Qty"] * 100).round(1)
df_alloc["SKU"]                 = SKU_CONTEXT
df_alloc["Total_Supply"]        = TOTAL_SUPPLY
df_alloc["Total_Demand"]        = TOTAL_DEMAND
df_alloc["Shortage_Cases"]      = SHORTAGE

assert df_alloc["Suggested_Cut_Qty"].sum()   == SHORTAGE,     "Cut total mismatch"
assert df_alloc["Final_Allocated_Qty"].sum() == TOTAL_SUPPLY, "Allocation total mismatch"

out_alloc = os.path.join(CURATED, "Intermediate_Allocation_Scenario.csv")
df_alloc.to_csv(out_alloc, index=False)

net_saved = df_alloc[df_alloc["Revenue_Saved"] > 0]["Revenue_Saved"].sum()

print(f"Scenario validated  |  Demand: {TOTAL_DEMAND:,}  Supply: {TOTAL_SUPPLY:,}  Shortage: {SHORTAGE:,}")
print(f"Flat cut rate      : {FLAT_CUT_RATE:.1%}")
print(f"Revenue protected  : ${net_saved:,.0f}  (Tier 1+2 vs flat-cut baseline)")
print(f"Saved -> {out_alloc}")
df_alloc

In [ ]:
# ── Cell 4 · Chart 1 – The Macro View ────────────────────────────────────────
# Business question: Which commodities are in shortage, and on which days?
#
# Grouped bar chart: daily Net_Position by commodity.
# Red dashed line at y=0 marks the shortage threshold.
# A bar below zero means that commodity is short on that date.

TEMPLATE = "plotly_white"
RED   = "#D7263D"
GREEN = "#2DC653"
BLUE  = "#1F77B4"
GREY  = "#BDBDBD"

TIER_COLORS = {
    "Tier 1 - High Margin":    "#1A6B3C",
    "Tier 2 - Standard":       "#F4A261",
    "Tier 3 - Flexible/Spot":  "#D7263D",
}

c1 = (
    df_viz[df_viz["Commodity"].notna()]
    .groupby(["Date", "Commodity"], as_index=False)
    .agg(Net_Position=("Net_Position", "sum"))
)

fig1 = px.bar(
    c1,
    x="Date",
    y="Net_Position",
    color="Commodity",
    barmode="group",
    title=(
        "<b>Chart 1 – Daily Net Position by Commodity</b><br>"
        "<sup>Negative bars = shortage risk  |  "
        "Planning horizon: 2026-03-16 to 2026-04-01</sup>"
    ),
    labels={"Net_Position": "Net Position (cases)", "Date": "Date"},
    template=TEMPLATE,
)
fig1.add_hline(
    y=0, line_dash="dash", line_color=RED, line_width=2,
    annotation_text="Shortage threshold",
    annotation_position="top left",
)
fig1.update_layout(
    hovermode="x unified",
    legend_title="Commodity",
    height=450,
)
fig1.show()

In [ ]:
# ── Cell 5 · Chart 2 – The Fire Alarm & Root Cause ───────────────────────────
# Business question: Why is RTBB1015 at MPL1 critically short?
#
# Two-panel chart:
#   Top    : Supply (bar) vs GoodSalesShipping demand (line) at MPL1 over time.
#             Red shading highlights shortage days (2026-03-22 to 2026-03-27).
#   Bottom : Inbound fill rate per location for RTBB1015 (received vs expected).
#
# Root cause: MPL2_INV and MPL6-B received 0 of 715 expected cases.
#             This drained MPL1 inventory from 1,319 (March 17) to 0 (March 23).

focal = (
    df_viz[
        (df_viz["LocationCode"] == "MPL1") &
        (df_viz["SKUCode"]      == "RTBB1015")
    ]
    .sort_values("Date")
    .copy()
)

# Inbound fill rate per location for RTBB1015 (SKUKey=4)
ib_rtbb = fact_ib[fact_ib["SKUKey"] == 4].merge(
    dim_loc[["LocationKey", "LocationCode"]], on="LocationKey", how="left"
)
ib_loc = (
    ib_rtbb
    .groupby("LocationCode", as_index=False)
    .agg(Expected=("ExpectedQty", "sum"), Received=("ReceivedQty", "sum"))
)
ib_loc["Fill_Rate_Pct"] = (
    ib_loc["Received"] / ib_loc["Expected"].replace(0, float("nan")) * 100
).round(1)
ib_loc["Bar_Color"] = ib_loc["Fill_Rate_Pct"].apply(
    lambda v: GREEN if v >= 90 else (BLUE if v >= 50 else RED)
)

fig2 = make_subplots(
    rows=2, cols=1,
    shared_xaxes=False,
    subplot_titles=(
        "RTBB1015 at MPL1 — Daily Supply vs Demand",
        "Inbound Fill Rate by Location — RTBB1015 (all locations)",
    ),
    row_heights=[0.60, 0.40],
    vertical_spacing=0.14,
)

# Panel 1: supply bar
fig2.add_trace(
    go.Bar(
        x=focal["Date"],
        y=focal["TotalSupply"],
        name="Total Supply",
        marker_color=BLUE,
        opacity=0.85,
        hovertemplate="<b>%{x|%Y-%m-%d}</b><br>Supply: %{y:,} cases<extra></extra>",
    ),
    row=1, col=1,
)
# Panel 1: demand line
fig2.add_trace(
    go.Scatter(
        x=focal["Date"],
        y=focal["GoodSalesShipping"],
        name="Ordered (GoodSalesShipping)",
        mode="lines+markers",
        line=dict(color=RED, width=2.5),
        marker=dict(size=7),
        hovertemplate="<b>%{x|%Y-%m-%d}</b><br>Ordered: %{y:,} cases<extra></extra>",
    ),
    row=1, col=1,
)
# Red shading on shortage days
for _, row_data in focal[focal["NetAvailable"] < 0].iterrows():
    fig2.add_vrect(
        x0=str(row_data["Date"].date()),
        x1=str((row_data["Date"] + pd.Timedelta(days=1)).date()),
        fillcolor=RED, opacity=0.09, line_width=0,
        row=1, col=1,
    )

# Panel 2: fill rate bars
fig2.add_trace(
    go.Bar(
        x=ib_loc["Fill_Rate_Pct"],
        y=ib_loc["LocationCode"],
        orientation="h",
        marker_color=ib_loc["Bar_Color"],
        text=ib_loc.apply(
            lambda r: f"{r['Fill_Rate_Pct']}%  "
                      f"({int(r['Received']):,} / {int(r['Expected']):,} cases)",
            axis=1,
        ),
        textposition="outside",
        hovertemplate="<b>%{y}</b><br>Fill Rate: %{x:.1f}%<extra></extra>",
        name="Inbound Fill Rate",
        showlegend=False,
    ),
    row=2, col=1,
)
fig2.add_vline(x=100, line_dash="dash", line_color="grey", row=2, col=1)
fig2.add_vline(
    x=53.1, line_dash="dot", line_color=RED, row=2, col=1,
    annotation_text="Overall 53%", annotation_font_size=9,
    annotation_position="bottom",
)

fig2.update_layout(
    title=dict(
        text=(
            "<b>Chart 2 – The Fire Alarm: RTBB1015 Supply Crisis at MPL1</b><br>"
            "<sup>Red shading = shortage days (2026-03-22 to 2026-03-27)  |  "
            "Root cause: MPL2_INV + MPL6-B received 0 of 715 expected cases</sup>"
        ),
        x=0,
    ),
    template=TEMPLATE,
    hovermode="x unified",
    height=580,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig2.update_xaxes(title_text="Fill Rate (%)", range=[0, 135], row=2, col=1)
fig2.update_yaxes(title_text="Cases", row=1, col=1)
fig2.show()

# Print root-cause summary
total_exp = ib_loc["Expected"].sum()
total_rec = ib_loc["Received"].sum()
print("RTBB1015 Inbound Summary:")
print(f"  Total Expected : {int(total_exp):,} cases")
print(f"  Total Received : {int(total_rec):,} cases  ({total_rec/total_exp*100:.1f}% fill rate)")
print(f"  Gap            : {int(total_exp - total_rec):,} cases NOT received (MPL2_INV + MPL6-B = 0%)")

In [ ]:
# ── Cell 5b · Data Prep – Overnight Exceptions ───────────────────────────────
#
# Simulates the 6:00 AM morning check a Category Manager performs every day.
# Instead of reading 40+ emails to find out what went wrong overnight, this
# cell filters Fact_Inbound for POs due TODAY that have NOT been received in
# full, building a precise exception list in seconds.
#
# TODAY_KEY is set to 20260317 (March 17, 2026).  In production this would be
# replaced with: int(pd.Timestamp.today().strftime("%Y%m%d"))
#
# Logic:
#   1. Filter Fact_Inbound to rows where DeliveryDateKey == TODAY_KEY.
#   2. Merge Dim_Location (LocationCode) and Dim_Vendor (VendorName).
#   3. Calculate Shortage_Variance = ExpectedQty - ReceivedQty.
#   4. Keep only rows where Shortage_Variance > 0 (missing / short shipments).
#   5. Aggregate to PO level (one bar per PO in the chart).
#   6. Assign a Severity_Band for colour coding.
#   7. Save to data/curated/Intermediate_Overnight_Exceptions.csv.

TODAY_KEY = 20_260_317   # simulated morning of 2026-03-17

# ── 5b.1  Filter to today's inbound ──────────────────────────────────────────
ib_today = fact_ib[fact_ib["DeliveryDateKey"] == TODAY_KEY].copy()
ib_today["Shortage_Variance"] = ib_today["ExpectedQty"] - ib_today["ReceivedQty"]

# Keep only exceptions (short / missing shipments)
ib_except = ib_today[ib_today["Shortage_Variance"] > 0].copy()

# ── 5b.2  Attach dimension labels ─────────────────────────────────────────────
dim_vend = pd.read_csv(os.path.join(CURATED, "Dim_Vendor.csv"))

ib_except = ib_except.merge(
    dim_loc[["LocationKey", "LocationCode", "LocationType"]],
    on="LocationKey", how="left"
)
ib_except = ib_except.merge(
    dim_vend[["VendorKey", "VendorName"]],
    on="VendorKey", how="left"
)
ib_except = ib_except.merge(
    dim_prod[["SKUKey", "SKUCode", "ProductDescription", "Commodity"]],
    on="SKUKey", how="left"
)

# ── 5b.3  Aggregate to PO level ───────────────────────────────────────────────
# One row per PO makes the chart readable; line-level detail is in the CSV.
po_exceptions = (
    ib_except
    .groupby(["PONumber", "VendorName", "LocationCode"], as_index=False)
    .agg(
        Lines            = ("SKUCode",           "count"),
        ExpectedQty      = ("ExpectedQty",        "sum"),
        ReceivedQty      = ("ReceivedQty",         "sum"),
        Shortage_Variance= ("Shortage_Variance",   "sum"),
        Commodities      = ("Commodity",           lambda x: ", ".join(sorted(x.dropna().unique()))),
    )
    .sort_values("Shortage_Variance", ascending=False)
    .reset_index(drop=True)
)

# ── 5b.4  Severity band (used for chart colour gradient) ─────────────────────
def severity(v):
    if v >= 2_000: return "Critical  (>=2,000)"
    if v >= 1_000: return "High      (1,000–1,999)"
    if v >=   500: return "Medium    (500–999)"
    return           "Low       (<500)"

po_exceptions["Severity_Band"] = po_exceptions["Shortage_Variance"].apply(severity)

# ── 5b.5  Build display label for Y-axis ─────────────────────────────────────
po_exceptions["PO_Label"] = (
    po_exceptions["PONumber"] + "  |  "
    + po_exceptions["VendorName"].str[:35]
    + "  [" + po_exceptions["LocationCode"] + "]"
)

# ── 5b.6  Save intermediate CSV ──────────────────────────────────────────────
out_exc = os.path.join(CURATED, "Intermediate_Overnight_Exceptions.csv")
po_exceptions.to_csv(out_exc, index=False)

total_missing = int(po_exceptions["Shortage_Variance"].sum())
print(f"Today ({TODAY_KEY}) Overnight Exception Summary:")
print(f"  POs with shortages : {len(po_exceptions):,}")
print(f"  Total cases missing: {total_missing:,}")
print(f"  Severity breakdown :")
print(po_exceptions["Severity_Band"].value_counts().to_string())
print(f"  Saved -> {out_exc}")
po_exceptions[["PONumber", "VendorName", "LocationCode", "ExpectedQty",
               "ReceivedQty", "Shortage_Variance", "Severity_Band"]]

In [ ]:
# ── Cell 5c · Chart 2.5 – The Overnight Detective ────────────────────────────
#
# Business question: Instead of reading 40 emails, which POs failed to arrive
#                    overnight — and how many cases are missing from each?
#
# Horizontal bar chart (one bar = one PO) sorted by Shortage_Variance (largest
# gap at the top).  Colour codes severity:
#   Dark red   = Critical  (>= 2,000 cases missing)
#   Red        = High      (1,000–1,999 cases missing)
#   Orange-red = Medium    (500–999 cases missing)
#   Amber      = Low       (< 500 cases missing)
#
# Story: The tallest red bar is the truck/grower the Category Manager must call
#        FIRST thing — before touching email.

SEVERITY_COLORS = {
    "Critical  (>=2,000)":    "#7B0D1E",   # deep crimson
    "High      (1,000–1,999)":"#D7263D",   # bright red
    "Medium    (500–999)":    "#F4622A",   # orange-red
    "Low       (<500)":       "#F4A261",   # amber
}

# Sort so largest bars appear at the top of the chart
df_exc = po_exceptions.sort_values("Shortage_Variance", ascending=True).copy()

total_missing_cases = int(df_exc["Shortage_Variance"].sum())
n_pos               = len(df_exc)

fig25 = px.bar(
    df_exc,
    x="Shortage_Variance",
    y="PO_Label",
    orientation="h",
    color="Severity_Band",
    color_discrete_map=SEVERITY_COLORS,
    category_orders={
        "Severity_Band": [
            "Critical  (>=2,000)",
            "High      (1,000–1,999)",
            "Medium    (500–999)",
            "Low       (<500)",
        ]
    },
    text="Shortage_Variance",
    title=(
        "<b>Chart 2.5 – Overnight Exceptions: Missing & Delayed Inbound Shipments</b><br>"
        f"<sup>As of 06:00 AM on 2026-03-17  |  "
        f"{n_pos} POs with shortfalls  |  "
        f"{total_missing_cases:,} total cases missing  |  "
        "Sorted by severity — call the top PO first</sup>"
    ),
    labels={
        "Shortage_Variance": "Cases Missing (Expected − Received)",
        "PO_Label":          "",
        "Severity_Band":     "Severity",
    },
    template=TEMPLATE,
    hover_data={
        "ExpectedQty":       True,
        "ReceivedQty":       True,
        "Shortage_Variance": True,
        "Commodities":       True,
        "PO_Label":          False,
        "Severity_Band":     False,
    },
)

fig25.update_traces(
    texttemplate="%{x:,} cases",
    textposition="outside",
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Expected  : %{customdata[0]:,} cases<br>"
        "Received  : %{customdata[1]:,} cases<br>"
        "Missing   : %{customdata[2]:,} cases<br>"
        "Commodities: %{customdata[3]}<extra></extra>"
    ),
)

# Vertical reference line at the average shortage per PO
avg_shortage = df_exc["Shortage_Variance"].mean()
fig25.add_vline(
    x=avg_shortage,
    line_dash="dot",
    line_color="grey",
    line_width=1.5,
    annotation_text=f"Avg: {avg_shortage:,.0f} cases",
    annotation_position="bottom right",
    annotation_font_size=10,
)

fig25.update_layout(
    height=max(460, len(df_exc) * 26 + 120),
    margin=dict(r=160, l=10),
    legend=dict(
        title="Severity",
        orientation="h",
        yanchor="bottom", y=1.01,
        xanchor="right",  x=1,
    ),
    xaxis=dict(title="Cases Missing (Expected − Received)"),
)
fig25.show()

# Actionable summary printed below the chart
print("Action List (top 5 POs to call immediately):")
top5 = po_exceptions.head(5)
for _, r in top5.iterrows():
    print(f"  {r['PONumber']:12s}  {r['VendorName'][:40]:40s}  "
          f"[{r['LocationCode']}]  {int(r['Shortage_Variance']):,} cases missing")

In [ ]:
# ── Cell 6 · Chart 3 – The Allocation Solution ────────────────────────────────
# Business question: How should the 2,000-case emergency resupply be allocated?
#
# Stacked overlay bar chart showing Original Order (grey) and Final Allocation
# (tier-coloured) per customer. Fill-rate percentages shown above each bar.
#
# Tier colour key:
#   Dark green = Tier 1 (High Margin)  → 100% fill rate
#   Amber      = Tier 2 (Standard)     → 100% fill rate
#   Red        = Tier 3 (Flexible/Spot)→  0% fill rate (deferred to next resupply)

tier_summary = (
    df_alloc.groupby("Priority_Tier", sort=False)
    .agg(
        Ordered   =("Original_Order_Qty",  "sum"),
        Allocated =("Final_Allocated_Qty", "sum"),
        Cut       =("Suggested_Cut_Qty",   "sum"),
    )
    .reset_index()
)

df_sorted = df_alloc.sort_values(
    ["Priority_Tier", "Original_Order_Qty"], ascending=[True, False]
)

fig3 = go.Figure()

# Grey backdrop = original orders
fig3.add_trace(go.Bar(
    name="Original Order",
    x=df_sorted["Customer_Name"],
    y=df_sorted["Original_Order_Qty"],
    marker_color=GREY,
    opacity=0.45,
    hovertemplate="<b>%{x}</b><br>Original Order: %{y:,} cases<extra></extra>",
))

# Coloured overlay = final allocation
fig3.add_trace(go.Bar(
    name="Final Allocation",
    x=df_sorted["Customer_Name"],
    y=df_sorted["Final_Allocated_Qty"],
    marker_color=[TIER_COLORS[t] for t in df_sorted["Priority_Tier"]],
    text=df_sorted["Fill_Rate_Pct"].astype(str) + "%",
    textposition="outside",
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Allocated: %{y:,} cases<br>"
        "Fill Rate: %{text}<extra></extra>"
    ),
))

fig3.update_layout(
    title=dict(
        text=(
            f"<b>Chart 3 – Strategic Allocation: "
            f"{TOTAL_SUPPLY:,} Emergency Cases Across {TOTAL_DEMAND:,} Backorders</b><br>"
            f"<sup>SKU: {SKU_CONTEXT}  |  "
            f"Shortage: {SHORTAGE:,} cases ({FLAT_CUT_RATE:.0%} flat cut)  |  "
            f"Tier 1 + Tier 2 fully protected  |  Tier 3 deferred to next resupply</sup>"
        ),
        x=0,
    ),
    barmode="overlay",
    yaxis_title="Cases",
    template=TEMPLATE,
    hovermode="x unified",
    height=490,
    xaxis_tickangle=-35,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig3.show()

# Tier allocation summary table
print("Tier Allocation Summary:")
print(tier_summary.to_string(index=False))

In [ ]:
# ── Cell 7 · Chart 4 – The Financial Impact ──────────────────────────────────
# Business question: What is the dollar value of tier-based vs flat-cut allocation?
#
# Two-panel layout:
#   Left  : Revenue saved / lost per customer vs flat-cut baseline (waterfall bars).
#           Green = revenue protected (Tier 1+2 gets more than flat cut).
#           Red   = revenue foregone  (Tier 3 gets less than flat cut).
#   Right : Grouped bar showing total revenue per tier: strategic vs flat-cut.
#
# Headline metric: strategic allocation protects $X more revenue than flat cut.

df_rev = df_alloc.sort_values("Revenue_Saved", ascending=True).copy()
df_rev["Bar_Color"] = df_rev["Revenue_Saved"].apply(
    lambda v: GREEN if v > 0 else (GREY if v == 0 else RED)
)
df_rev["Label"] = df_rev["Revenue_Saved"].apply(
    lambda v: f"+${v:,.0f}" if v >= 0 else f"-${abs(v):,.0f}"
)

tier_rev = (
    df_alloc
    .groupby("Priority_Tier", sort=False)
    .apply(lambda g: pd.Series({
        "Strategic_Revenue": (g["Final_Allocated_Qty"] * g["Unit_Price"]).sum(),
        "FlatCut_Revenue":   (g["Flat_Allocated_Qty"]  * g["Unit_Price"]).sum(),
    }))
    .reset_index()
)

strat_total = tier_rev["Strategic_Revenue"].sum()
flat_total  = tier_rev["FlatCut_Revenue"].sum()

fig4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Revenue vs Flat-Cut Baseline per Customer",
        "Revenue by Tier: Strategic vs Flat Cut",
    ),
    column_widths=[0.55, 0.45],
    horizontal_spacing=0.10,
)

# Left panel: per-customer waterfall
fig4.add_trace(
    go.Bar(
        x=df_rev["Revenue_Saved"],
        y=df_rev["Customer_Name"],
        orientation="h",
        marker_color=df_rev["Bar_Color"],
        text=df_rev["Label"],
        textposition="outside",
        hovertemplate="<b>%{y}</b><br>Revenue vs flat-cut: $%{x:,.0f}<extra></extra>",
    ),
    row=1, col=1,
)
fig4.add_vline(
    x=0, line_color="black", line_width=1.5,
    annotation_text="Flat baseline", annotation_position="top",
    row=1, col=1,
)

# Right panel: grouped tier bars
fig4.add_trace(
    go.Bar(
        name="Strategic Allocation",
        x=tier_rev["Priority_Tier"],
        y=tier_rev["Strategic_Revenue"],
        marker_color=[TIER_COLORS[t] for t in tier_rev["Priority_Tier"]],
        text=tier_rev["Strategic_Revenue"].apply(lambda v: f"${v:,.0f}"),
        textposition="outside",
        hovertemplate="<b>%{x}</b><br>Strategic: $%{y:,.0f}<extra></extra>",
    ),
    row=1, col=2,
)
fig4.add_trace(
    go.Bar(
        name="Flat-Cut Baseline",
        x=tier_rev["Priority_Tier"],
        y=tier_rev["FlatCut_Revenue"],
        marker_color=GREY,
        marker_pattern_shape="/",
        text=tier_rev["FlatCut_Revenue"].apply(lambda v: f"${v:,.0f}"),
        textposition="outside",
        hovertemplate="<b>%{x}</b><br>Flat cut: $%{y:,.0f}<extra></extra>",
    ),
    row=1, col=2,
)

fig4.update_layout(
    title=dict(
        text=(
            f"<b>Chart 4 – Financial Impact: Strategic vs Flat-Cut Allocation</b><br>"
            f"<sup>Strategic total: ${strat_total:,.0f}  |  "
            f"Flat-cut total: ${flat_total:,.0f}  |  "
            f"Net revenue protected: ${strat_total - flat_total:,.0f}</sup>"
        ),
        x=0,
    ),
    template=TEMPLATE,
    height=510,
    barmode="group",
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    margin=dict(r=130),
    yaxis={"autorange": "reversed"},
)
fig4.update_xaxes(title_text="Revenue vs Flat-Cut Baseline ($)", row=1, col=1)
fig4.update_yaxes(title_text="Revenue ($)", row=1, col=2)
fig4.show()

print("Revenue Summary:")
print(f"  Strategic allocation total : ${strat_total:,.0f}")
print(f"  Flat-cut baseline total    : ${flat_total:,.0f}")
print(f"  Net revenue protected      : ${strat_total - flat_total:,.0f}")